In [ ]:
import pandas as pd
import json
import ast
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
import seaborn as sns
import numpy as np
from scipy.optimize import curve_fit
from tqdm import tqdm
from tqdm.auto import tqdm
tqdm.pandas()
import os
import math
import psutil
import gc
import sqlite3
from sklearn.metrics import mean_squared_error
from scipy.interpolate import UnivariateSpline
from sklearn.metrics import r2_score
from scipy import stats



df_moves = pd.read_parquet(r'..\df_moves6.parquet')

# df_moves = df_moves.drop(columns=['E1', 'E2', 'E3', 'E4', 'E5'])

In [ ]:
df_moves.columns

In [ ]:
df_moves.head()

### Summary of Fitting Parameters

| ELO Group | A | τ | x₀ | A_err | τ_err | x₀_err | Max CI Width | d_max | min_cnt |
|-----------|------|------|------|------|------|------|------|------|------|
| ≥3000 | 0.874 | 167.4 | -9.7 | 0.0023 | 2.81 | 1.85 | 0.0107 | 700 | 50 |
| 2800–3000 | 0.867 | 175.5 | 4.0 | 0.0018 | 2.16 | 1.29 | 0.0077 | 700 | 50 |
| 2600–2800 | 0.857 | 174.9 | 23.2 | 0.0017 | 1.83 | 1.06 | 0.0064 | 700 | 80 |
| 2400–2600 | 0.848 | 176.2 | 39.6 | 0.0018 | 1.84 | 1.04 | 0.0067 | 700 | 100 |
| 2200–2400 | 0.838 | 178.3 | 52.5 | 0.0021 | 2.08 | 1.18 | 0.0075 | 700 | 100 |
| 2000–2200 | 0.829 | 176.8 | 66.3 | 0.0029 | 2.77 | 1.61 | 0.0095 | 700 | 50 |
| <2000 | 0.842 | 184.8 | 89.3 | 0.0075 | 5.49 | 3.52 | 0.0166 | 700 | 50 |    


# 关于Forced Move的处理
## Δ & S & P = NaN

In [ ]:
forced_count = (df_moves['Is_Forced'] == 1).sum()
print(f'数据集中所有强制走子的数量为{forced_count}')

In [ ]:
cols_to_check = ['P_model', 'S', 'P_model(Global=10.0cp)', 'S(Global=10.0cp)', 'Δ']
cols_to_check = [c for c in cols_to_check if c in df_moves.columns]

print("================ 异常值统计 ================")
for col in cols_to_check:
    # 统计 NaN 的数量
    nan_count = df_moves[col].isna().sum()
    # 统计 Inf 或 -Inf 的数量
    inf_count = np.isinf(df_moves[col]).sum()
    print(f"列 [{col}]:\t NaN 数量 = {nan_count:,} \t Inf 数量 = {inf_count:,}")

# 用 Group Specific 的 P_model 和 S 来做基础过滤 提取包含异常值的行
mask_anomalies = (
    df_moves['P_model'].isna() | np.isinf(df_moves['P_model']) |
    df_moves['S'].isna() | np.isinf(df_moves['S'])
)

df_anomalies = df_moves[mask_anomalies]

print(f"\n================ 异常值数据采样 ================")
print(f"总共发现 {len(df_anomalies):,} 行带有异常值的记录。")

if len(df_anomalies) > 0:
    display_cols = ['uid', 'Move_Idx', 'ELO', 'ELO_Group', 'Δ', 'A', 'P_model', 'S', 'Is_Forced']
    display_cols = [c for c in display_cols if c in df_anomalies.columns]
    
    print("\n前 15 行异常数据：")
    print(df_anomalies[display_cols].head(35))

In [ ]:
(df_moves['Progress'] == 1).sum()

In [ ]:
cols = ['uid','Move_Idx','Is_Error', 'Is_Blunder', 'Progress','Δ', 'Δi', 'P_model', 'S', 'Is_Forced', 'Is_Optimal']
cols = [c for c in cols if c in df_moves.columns]

df_moves[(df_moves['Progress'] == 1) & (df_moves['Is_Forced'] == 1)][cols].head(20)

# 画S P 的分布 （排除了强制走子）

In [ ]:
GLOBAL_τ_List = [10.0, 50.0, 100.0, 150.0]
not_forced_mask = (df_moves['Is_Forced'] != 1).to_numpy()

# 计算总行数
n_rows = len(GLOBAL_τ_List) + 1
n_cols = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
# 列名匹配函数
def get_column_name(df, base_name, τ):
    col_float = f'{base_name}(Global={τ}cp)'
    col_int = f'{base_name}(Global={int(τ)}cp)'
    return col_float if col_float in df.columns else col_int


def fast_hist(ax, data, bins=30, color=None):
    counts, edges = np.histogram(data, bins=bins)
    prob = counts / counts.sum()
    width = np.diff(edges)
    ax.bar(edges[:-1], prob, width=width, align="edge", color=color, edgecolor="black", linewidth=0.6)

# Global τ
for i, τ in enumerate(GLOBAL_τ_List):

    p_col = get_column_name(df_moves, 'P_model', τ)
    s_col = get_column_name(df_moves, 'S', τ)

    if s_col not in df_moves.columns:
        print(f"跳过: 找不到列 {s_col}")
        continue

    # P
    if p_col in df_moves.columns:
        data = df_moves[p_col].to_numpy()[not_forced_mask]
        fast_hist(axes[i,0], data)
        axes[i,0].set_title(
            f'Global P_model (τ={τ}cp)',
            fontsize=14,
            fontweight='bold'
        )
        axes[i,0].set_xlabel('P_model')
        axes[i,0].set_ylabel('Proportion')
    # S
    if s_col in df_moves.columns:
        data = df_moves[s_col].to_numpy()[not_forced_mask]
        fast_hist(axes[i,1], data, color='orange')
        axes[i,1].set_title(
            f'Global Difficulty S (τ={τ}cp)',
            fontsize=14,
            fontweight='bold'
        )
        axes[i,1].set_xlabel('S (Bits)')
        axes[i,1].set_ylabel('Proportion')
# Group Specific
last_row_idx = n_rows - 1
if 'P_model' in df_moves.columns:
    data = df_moves['P_model'].to_numpy()[not_forced_mask]
    fast_hist(axes[last_row_idx,0], data)
    axes[last_row_idx,0].set_title(
        'Group Specific: P_model (Optimized)',fontsize=14,fontweight='bold',color='darkblue')
    axes[last_row_idx,0].set_xlabel('P_model')
    axes[last_row_idx,0].set_ylabel('Proportion')

    data = df_moves['S'].to_numpy()[not_forced_mask]
    fast_hist(axes[last_row_idx,1], data, color='orange')
    axes[last_row_idx,1].set_title('Group Specific: Difficulty S (Optimized)',fontsize=14,fontweight='bold',color='darkblue')
    axes[last_row_idx,1].set_xlabel('S (Bits)')
    axes[last_row_idx,1].set_ylabel('Proportion')


plt.tight_layout()
plt.show()

### 解释新的P_model为什么会有<0.5的值
以 <2000 分的低分段组为例，他们的 $x_0 = 91.5$。    
假设这个玩家当前面对一个极为基础的局面（即复杂度 $\Delta = 0$），代入公式看看会发生什么：     
$-(\Delta - x_0) = -(0 - 91.5) = 91.5$      
指数部分变成了正数：$91.5 / 181.3 \approx 0.505$$e^{0.505} \approx 1.657$     
分母变成了 $1 + 1.657 = 2.657$    
最终 $P_{model} = 0.855 / 2.657 \approx$ 0.32

# P_model 和 P_emp 的关系曲线  （排除了强制走子）

In [ ]:
elo_order=['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']
# distinct_palette = sns.color_palette("Set1", len(elo_order))
distinct_palette = sns.color_palette("viridis", len(elo_order))
# 拟合结果
elo_params = {
    '>=3000':      {'A': 0.874, 'τ': 167.4, 'x0': -9.7},
    '2800-3000':   {'A': 0.867, 'τ': 175.5, 'x0': 4.0},
    '2600-2800':   {'A': 0.857, 'τ': 174.9, 'x0': 23.2},
    '2400-2600':   {'A': 0.848, 'τ': 176.2, 'x0': 39.6},
    '2200-2400':   {'A': 0.838, 'τ': 178.3, 'x0': 52.5},
    '2000-2200':   {'A': 0.829, 'τ': 176.8, 'x0': 66.3},
    '<2000':       {'A': 0.842, 'τ': 184.8, 'x0': 89.3}
}

def plot_delta_performance(data, delta_col, title, ax, tau_val=None, is_group_specific=False):
    """
    支持单条 Global 参考线或多条 Elo-specific 参考线
    """
    # 高速聚合
    plot_df = data[(data[delta_col] >= 0) & (data[delta_col] <= 800)]
    plot_df['Delta_Bin'] = (plot_df[delta_col] // 15) * 15
    
    agg_data = (plot_df.groupby(['ELO_Group', 'Delta_Bin'])['Is_Optimal']
                .mean().reset_index()
                .rename(columns={'Is_Optimal': 'P_emp'}))

    # 实测数据
    for i, group in enumerate(elo_order):
        group_data = agg_data[agg_data['ELO_Group'] == group]
        color = distinct_palette[i]
        # 实测散点
        # ax.scatter(group_data['Delta_Bin'], group_data['P_emp'], color=color, s=15, alpha=0.3, edgecolor='none')
        # 实测平滑趋势线
        sns.lineplot(data=group_data, x='Delta_Bin', y='P_emp', 
                     color=color, linewidth=3.0, ax=ax, 
                     label=group if is_group_specific else None)

    # 理论参考线
    x_ref = np.linspace(0, 800, 100)
    
    if is_group_specific:
        # 画理论曲线
        for i, group in enumerate(elo_order):
            A = elo_params[group]['A']
            tau = elo_params[group]['τ']
            x0 = elo_params[group]['x0']

            y_ref = A / (1 + np.exp(-(x_ref - x0) / tau))
            ax.plot(x_ref,y_ref,color='grey',linestyle='--',alpha=0.5,linewidth=2)

        ax.set_title(title, fontsize=12, fontweight='bold')
        # 求最上和最下曲线位置 写图例
        A_top = elo_params[elo_order[0]]['A']
        tau_top = elo_params[elo_order[0]]['τ']
        x0_top = elo_params[elo_order[0]]['x0']
        A_bottom = elo_params[elo_order[-1]]['A']
        tau_bottom = elo_params[elo_order[-1]]['τ']
        x0_bottom = elo_params[elo_order[-1]]['x0']
        label_x = x_ref[-1] + 20
        y_top = A_top / (1 + np.exp(-(label_x - x0_top) / tau_top))
        y_bottom = A_bottom / (1 + np.exp(-(label_x - x0_bottom) / tau_bottom))
        # 写 L1 和 L6
        ax.text(label_x, y_top, "L1", fontsize=10, ha='left', va='bottom')
        ax.text(label_x, y_bottom, "L6", fontsize=10, ha='left', va='top')
        # 画箭头
        ax.annotate( '', xy=(label_x, y_bottom),xytext=(label_x, y_top), arrowprops=dict(arrowstyle='->', color='grey', linewidth=1.5))
            
    else:
        # 模式 B: Global 探测模式，使用单一黑色参考线，分子 A = 1
        y_ref = 1 / (1 + np.exp(-x_ref / tau_val))
        ax.plot(x_ref, y_ref, 'k--', alpha=0.3, linewidth=2.5, 
                label=f'Global Model ($\hat{{P}}$ @ $\\tau$={tau_val})')
        ax.set_title(title, fontsize=12, fontweight='bold')

    # 样式设置
    ax.set_xlabel(r'Decision Margin Δ_top5 (cp)', fontsize=10)
    ax.set_ylabel(r'$P_{emp}$', fontsize=12)
    ax.set_ylim(0.2, 1.0)
    ax.grid(True, linestyle=':', alpha=0.3)
    if is_group_specific:
        ax.legend(loc='upper left')

fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(3, 2)

# 第一个格子: Group-specific，展示 7 条拟合出的模型线
ax0 = fig.add_subplot(gs[0, 0])
plot_delta_performance(df_moves, 'Δ', 'Specific Calibration', ax0, is_group_specific=True)

# 随后的格子: Global 探测
for idx, tau in enumerate(GLOBAL_τ_List[:4]):
    row, col = (idx + 1) // 2, (idx + 1) % 2
    ax = fig.add_subplot(gs[row, col])
    plot_delta_performance(df_moves, 'Δ', f'Global $\\tau$ = {tau}cp', ax, tau_val=tau)

# 最后一个格子: 术语与公式说明
ax_legend = fig.add_subplot(gs[2, 1])
ax_legend.axis('off')
legend_text = (
    "Definitions & Annotations:\n\n"
    "• P_emp: Empirical probability of selecting the engine-recommended move\n"
    "• Δ_top5 (cp): The Centipawn gap between the E1 and E2-E5.\n\n"
    "• Solid Line: Observed performance trend for each ELO group.\n\n"
    "• Dashed Line (Group): Theoretical model with optimized parameters:\n"
    "  P_model = A / (1 + exp(-(Δ - x0) / τ)), where A, τ, and x0 are group-specific.\n\n"
    "• Dashed Line (Global): Reference curve with theoretical formula:\n"
    "  P_model = 1 / (1 + exp(-Δ / τ)) for comparison."
)
ax_legend.text(0.05, 0.5, legend_text, fontsize=11, transform=ax_legend.transAxes, 
               va='center', linespacing=1.8, bbox=dict(facecolor='none', edgecolor='gray', alpha=0.15))

plt.tight_layout()
plt.show()

实测数据在 Δ 接近 0 时准确率极高，随后迅速下降，这在国际象棋中被称为**“简单局面悖论”**：
 
高准确率起点 ( Δ 接近 0 )：当 Δ 非常小时，意味着局面处于“非关键期”。此时最好的两步棋评分几乎一样，根据你的定义（容差 $10cp$），玩家只要不走错得离谱，几乎随便走一步都是“最优”的 。此外，这也包含了大量的开局库（Book moves）和强制招法（Forced moves），这些棋步几乎不需要思考就能走对 。     

触底下降区：随着 Δ 略微增大，局面进入“关键期（Critical positions）”。此时正确的路径只有一条，而替代方案虽然只差一点点，但已经足以致命。人类在面对这种“微妙但重要”的差异时最容易犯错，因此准确率降至最低 。     

逐步回升区：当 Δ 变得很大（如 $200cp$ 以上）时，最优解变得非常“直观”（例如白捡一个子），此时准确率才开始遵循你的理论模型，随难度的降低而回升 。

# Cog_Speed distribution    
### 耗时为0的步子，认知速度设为了inf    
### 画图的时候舍弃被设为inf的走子，同时只话Is_Optimal = 1的走子

In [ ]:
mask = (df_moves['Is_Optimal'] == 1) & np.isfinite(df_moves['Cog_Speed'])
speed_data = df_moves.loc[mask, 'Cog_Speed']

plt.figure(figsize=(5,3))
plt.hist(speed_data, bins=50, edgecolor='black', alpha=0.8)
plt.xlabel('Cog_Speed')
plt.ylabel('Count')
plt.title('Distribution of Cog_Speed (Is_Optimal = 1, drop Move_Time = 0)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

counts, edges = np.histogram(speed_data, bins = 50)
for i in range(len(counts)):
    print(f"Bin: {edges[i]:.2f}  -  {edges[i+1]:.2f}  Counts: {counts[i]}")

del mask
gc.collect()

In [ ]:
speed_focus0 = speed_data[speed_data <= 5]
# 0-8
plt.figure(figsize=(5,3))
plt.hist(speed_focus0, bins=80, edgecolor='black', alpha=0.8)
plt.xlabel('Cog_Speed')
plt.ylabel('Count')
plt.title('Distribution of Cog_Speed (0–200)')
plt.grid(alpha=0.3)
plt.show()

del speed_focus0
gc.collect()

mask = (df_moves['Is_Optimal'] == 1) & np.isfinite(df_moves['Cog_Speed'])  的结果    
但是是否排除check/checkmate走子对这个图影响不大；
再多排除force走子之后末尾的认知速度稍有提升，但还是不特别大

In [ ]:
elo_order=['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']

distinct_palette = sns.color_palette("viridis", len(elo_order))
color_map = dict(zip(elo_order, distinct_palette))

mask = (df_moves['Is_Optimal'] == 1) & np.isfinite(df_moves['Cog_Speed']) # &(df_moves['Is_check'] == 0) & (df_moves['Is_checkmate'] == 0) &(df_moves['Is_Forced']==0)
time_complex = df_moves[mask].copy()
print(f"过滤前行数: {len(df_moves[(df_moves['Is_Optimal'] == 1) & np.isfinite(df_moves['Cog_Speed'])])}")
print(f"过滤后行数: {len(time_complex)}")

# 分箱
time_complex['Progress_Bin'] = (time_complex['Progress'] * 100).astype(int)

agg_metrics = time_complex.groupby(['ELO_Group', 'Progress_Bin'])['Cog_Speed'].agg(
    mean='mean'
).reset_index()

fig, ax = plt.subplots(figsize=(15, 8))
for group in elo_order:
    g_data = agg_metrics[agg_metrics['ELO_Group'] == group]
    ax.plot(g_data['Progress_Bin'], g_data['mean'], 
             color=color_map[group], linewidth=2.5, label=group)

ax.set_title(r'Mean Cognitive Speed over Game Progress', fontsize=16, fontweight='bold')
ax.set_xlabel('Game Progress (%)', fontsize=13)  # 【新增】：把原本在下面那张图的 X 轴标签补上来
ax.set_ylabel('Mean Speed', fontsize=13)
# 均值天花板设为 10
ax.set_ylim(0, 6)  
ax.grid(True, linestyle=':', alpha=0.4)

ax.legend(title='ELO Group', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
for group in elo_order:
    g_data = agg_metrics[agg_metrics['ELO_Group'] == group].sort_values('Progress_Bin')
    # 平滑，但也会丢失数据真实性
    # smooth = g_data['mean'].rolling(window=3, center=True).mean()
    ax.plot(g_data['Progress_Bin'], g_data['mean'], 
             color=color_map[group], linewidth=2.5, label=group)
ax.set_title(r'Mean Cognitive Speed over Game Progress', fontsize=16, fontweight='bold')
ax.set_xlabel('Game Progress (%)', fontsize=13)  # 【新增】：把原本在下面那张图的 X 轴标签补上来
ax.set_ylabel('Mean Speed', fontsize=13)
# 均值天花板设为 10
ax.set_ylim(0.2, 2.4)  
ax.grid(True, linestyle=':', alpha=0.4)

ax.legend(title='ELO Group', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Remain Time和Cog_Speed的关系
#### 时间压力到底是激发人们思考更快还是会降低认知速度？在什么区间下压力反而带来正向影响？

In [ ]:
mask = (df_moves['Is_Optimal'] == 1) & np.isfinite(df_moves['Cog_Speed'])
time_complex = df_moves[mask]

# 对 Remain_Time 每 10 秒一个箱子
bin_size = 10
time_complex['Remain_Time_Bin'] = (time_complex['Remain_Time'] // bin_size) * bin_size
# 分组求聚合指标 (mean/median)
agg_metrics = time_complex.groupby(['ELO_Group', 'Remain_Time_Bin'])['Cog_Speed'].agg(
    speed_val='mean'  # 也可以选median
).reset_index()

distinct_palette = sns.color_palette("viridis", len(elo_order))
color_map = dict(zip(elo_order, distinct_palette))
fig, ax = plt.subplots(figsize=(15, 8))

for group in elo_order:
    g_data = agg_metrics[agg_metrics['ELO_Group'] == group]
    # 180s
    g_data = g_data[g_data['Remain_Time_Bin'] <= 180] 
    ax.plot(g_data['Remain_Time_Bin'], g_data['speed_val'], 
             color=color_map[group], linewidth=2.5, label=group)

ax.set_title(r'Median Cognitive Speed under Time Pressure', fontsize=16, fontweight='bold')
ax.set_xlabel('Remaining Time in Seconds (Rich $\longrightarrow$ Tense)', fontsize=13)
ax.set_ylabel('Median Cognitive Speed (ECP)', fontsize=13)

# 反转 X 轴：从剩余很多时间到很少时间
ax.invert_xaxis()

# 根据 median 的真实分布限制 Y 轴。如果是 median，最高大概在 0.5 到 1.5 之间
ax.set_ylim(0, 3.0)  
ax.grid(True, linestyle=':', alpha=0.4)

ax.legend(title='ELO Group', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

开局阶段，时间还省得很多且可以背谱，所有选手普遍思考偏快且几乎怎么走都算最优解，因而分数偏高。    
游戏进入中局时局面变难且时间还算充裕，因而更偏向慢慢思考，高分段选手的优势逐渐显现    
残局阶段，由于时间压力，选手思考时间被压缩，潜力得到了激发    

In [ ]:
del mask
del time_complex
gc.collect()

感觉还需要在这里立马插入横坐标为progress 纵坐标分别为Δ_top5 和 思考时间的图         
横坐标为Δ_top5 在 figure1.ipynb里

In [ ]:
df_moves.columns

In [ ]:
df_moves['Move_Time'].describe()

In [ ]:
mask = df_moves['Is_Forced'] != 1
plot_df = df_moves[mask]

# 对 Progress 进行分箱
plot_df['Progress_Bin'] = (plot_df['Progress'] * 100).astype(int)

# 按 ELO 分组和进度分箱，平均 Move_Time
agg_trends = plot_df.groupby(['ELO_Group', 'Progress_Bin'])['Move_Time'].agg(
    mean_time='median'  #均值受极端值影响太大，改成中位数了
).reset_index()

distinct_palette = sns.color_palette("viridis", len(elo_order))
color_map = dict(zip(elo_order, distinct_palette))
fig, ax = plt.subplots(figsize=(15, 8))
for group in elo_order:
    g_data = agg_trends[agg_trends['ELO_Group'] == group]
    # 排序确保线条从开局到残局连续
    g_data = g_data.sort_values('Progress_Bin')  
    ax.plot(g_data['Progress_Bin'], g_data['mean_time'], 
            color=color_map[group], linewidth=2.5, label=group)

ax.set_title('Median Thinking Time over Game Progress', fontsize=16, fontweight='bold')
ax.set_xlabel('Game Progress (%)', fontsize=13)
ax.set_ylabel('Mean Move Time (seconds)', fontsize=13)
# 限制 Y 轴范围：为了看清主体趋势，避免设太高会压扁曲线
ax.set_ylim(0, plot_df['Move_Time'].mean() * 2) 

ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(title='ELO Group', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
del mask
gc.collect()

In [ ]:
# Move_Time 列值为 0 的比例
(df_moves['Move_Time']==0).mean()

## Complexity to Blunder/Mistake

In [ ]:
def plot_error_rate_by_progress_fast(df, bins=50):    
    # 分箱
    bin_indices = np.clip((df['Progress'].values * bins).astype(int), 0, bins - 1)
    # 仅提取要用的数据
    temp_df = pd.DataFrame({'ELO_Group': df['ELO_Group'].values, 'Bin_Idx': bin_indices, 'Is_Error': df['Is_Error'].values})
    
    # 各 ELO 组的统计量
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Is_Error'].agg(['sum', 'count']).reset_index()
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    group_stats['Progress_Mid'] = (group_stats['Bin_Idx'] + 0.5) / bins

    # 全局 (Global) 的统计量， 用已聚合的 group_stats 二次聚合
    global_stats = group_stats.groupby('Bin_Idx')[['sum', 'count']].sum().reset_index()
    global_stats['Error_Rate'] = global_stats['sum'] / global_stats['count']
    global_stats['Progress_Mid'] = (global_stats['Bin_Idx'] + 0.5) / bins

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    # 全局均线
    ax.plot(global_stats['Progress_Mid'], global_stats['Error_Rate'], 
        color='black', linestyle='--', linewidth=3, alpha=0.7, label='Global Average',zorder=3)
    # ELO 分组线
    for i, group in enumerate(elo_order):
        group_data = group_stats[group_stats['ELO_Group'] == group]
        group_data = group_data.sort_values('Progress_Mid')
        
        ax.plot(group_data['Progress_Mid'],group_data['Error_Rate'], 
            color=distinct_palette[i], linewidth=1.5, alpha=0.85, label=group,zorder=2)
    
    plt.title("Error Rate Across Game Progress", fontsize=14, fontweight='bold')
    plt.xlabel("Game Progress (0.0=Start, 1.0=End)", fontsize=12)
    plt.ylabel("P(Error)", fontsize=12)
    plt.xlim(0, 1)
    max_y = max(group_stats['Error_Rate'].max(), global_stats['Error_Rate'].max())
    plt.ylim(0, max_y * 1.15) 
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9)
    text_str = "P(Error) = Count(Mistake/Blunder) / Count(Total)"
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.98, 0.03, text_str, transform=ax.transAxes, fontsize=11,verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

plot_error_rate_by_progress_fast(df_moves, bins=100)

In [ ]:
df_moves.columns

In [ ]:
%whos

In [ ]:
del data
del speed_data
gc.collect()

In [ ]:
# 去掉强制走子
def plot_error_rate_final_resort(df, bins=100, elo_order=None, distinct_palette=None):
    """
    针对 8800万行数据的究极优化：零切片，直接聚合
    """
    print(f"当前处理数据规模: {len(df):,} 行")

    # --- 1. 预处理：确保类型最省内存 (原地修改) ---
    # 如果这些列已经是正确类型，这些操作很快
    if df['Is_Error'].dtype != 'int8': df['Is_Error'] = df['Is_Error'].astype('int8')
    if df['Is_Forced'].dtype != 'int8': df['Is_Forced'] = df['Is_Forced'].astype('int8')
    
    # --- 2. 计算分箱指数 (得到一个独立的 numpy 数组) ---
    # 不要把这个数组加到 df 里，直接作为 groupby 的键
    print("正在计算分箱索引...")
    bin_idx = (df['Progress'].values * bins).astype(np.int32)
    np.clip(bin_idx, 0, bins - 1, out=bin_idx)

    # --- 3. 核心：直接全量聚合 ---
    # 我们同时按 ELO_Group, bin_idx 和 Is_Forced 分组
    # 这样整个过程不需要对大表做过滤，也就不会产生 MemoryError
    print("正在执行全量聚合 (不进行预过滤)...")
    aggregated = df.groupby(
        [df['ELO_Group'], bin_idx, df['Is_Forced']], 
        observed=True
    )['Is_Error'].agg(['sum', 'count']).reset_index()

    # 聚合完的一瞬间，大表使命完成。重命名列名
    aggregated.columns = ['ELO_Group', 'Bin_Idx', 'Is_Forced', 'sum', 'count']

    # --- 4. 剔除“强制走子”的数据 ---
    # 此时 aggregated 只有几百行，随你怎么过滤都不会炸内存
    print("正在清理小规模聚合数据...")
    group_stats = aggregated[aggregated['Is_Forced'] != 1].copy()
    
    # 按照之前的逻辑计算比率
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    group_stats['Progress_Mid'] = (group_stats['Bin_Idx'] + 0.5) / bins

    # 全局统计量
    global_stats = group_stats.groupby('Bin_Idx')[['sum', 'count']].sum().reset_index()
    global_stats['Error_Rate'] = global_stats['sum'] / global_stats['count']
    global_stats['Progress_Mid'] = (global_stats['Bin_Idx'] + 0.5) / bins

    # --- 5. 绘图 (此时内存非常安全) ---
    print("正在生成可视化...")
    plt.figure(figsize=(12, 7))
    # ... (后续绘图代码与之前完全一致) ...
    ax = plt.gca()
    ax.plot(global_stats['Progress_Mid'], global_stats['Error_Rate'], color='black', linestyle='--', linewidth=3, label='Global Average')
    
    if elo_order is None: elo_order = group_stats['ELO_Group'].unique()
    for i, group in enumerate(elo_order):
        subset = group_stats[group_stats['ELO_Group'] == group].sort_values('Progress_Mid')
        color = distinct_palette[i] if (distinct_palette and i < len(distinct_palette)) else None
        ax.plot(subset['Progress_Mid'], subset['Error_Rate'], linewidth=1.5, alpha=0.8, label=group, color=color)

    plt.title(f"Error Rate Analysis (N={len(df):,})")
    plt.xlabel("Game Progress"); plt.ylabel("P(Error)")
    plt.legend(loc='upper left', ncol=2); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    del aggregated, group_stats, global_stats
    gc.collect()

# --- 直接调用，千万不要传过滤后的 df ---
plot_error_rate_final_resort(df_moves, bins=100, elo_order=elo_order, distinct_palette=distinct_palette)

In [ ]:
# 去掉强制走子 check checkmate， 结尾有一些不一样了
musk = df_moves[(df_moves['Is_Forced'] != 1) & (df_moves['Is_check'] != 1) & (df_moves['Is_checkmate'] != 1)]
plot_error_rate_by_progress_fast(musk, bins=100)
del musk
gc.collect()

In [ ]:
def plot_error_rate_with_tau_markers(df, delta_col='Δ', max_delta=500, bins=20): 
    # 极速分箱逻辑保持不变
    vals = np.clip(df[delta_col].values, 0, max_delta)
    bin_indices = np.clip((vals / max_delta * bins).astype(int), 0, bins - 1)
    
    temp_df = pd.DataFrame({
        'ELO_Group': df['ELO_Group'].values, 
        'Bin_Idx': bin_indices, 
        'Is_Error': df['Is_Error'].values
    })
    
    group_stats = temp_df.groupby(['ELO_Group', 'Bin_Idx'], observed=False)['Is_Error'].agg(['sum', 'count']).reset_index()
    group_stats['Error_Rate'] = group_stats['sum'] / group_stats['count']
    bin_width = max_delta / bins
    group_stats['Delta_Mid'] = (group_stats['Bin_Idx'] + 0.5) * bin_width

    plt.figure(figsize=(12, 7))
    ax = plt.gca()
    
    # 画 ELO 分组线，并在 tau 处打点
    for i, group in enumerate(elo_order):
        group_data = group_stats[group_stats['ELO_Group'] == group]
        group_data = group_data.sort_values('Delta_Mid')
        color = distinct_palette[i]
        # tau_val = τ_map[group]
        
        # 画主线
        ax.plot(group_data['Delta_Mid'], group_data['Error_Rate'], 
                color=color, linewidth=1.8, alpha=0.8, label=f"{group}")

    plt.title(f"Error Rate Across {delta_col}_top5", fontsize=14, fontweight='bold')
    plt.xlabel(f"{delta_col}_top5 (cp)", fontsize=12)
    plt.ylabel("P(Error)", fontsize=12)
    
    plt.xlim(0, max_delta)
    # 只看 500cp 以内，所以 Y 轴最高基本在 0.25 左右，动态截取让图更好看
    max_y = group_stats['Error_Rate'].max()
    plt.ylim(0, max_y * 1.1) 
    
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9, title='ELO Group')
    
    # text_str = "Circular markers indicate the calibrated $\\tau$ value for each group."
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.8)
    # ax.text(0.98, 0.90, transform=ax.transAxes, fontsize=11, verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)

    plt.tight_layout()
    plt.show()

# 横轴截断，以观察看最核心的决策区
# plot_error_rate_with_tau_markers(complex_drop_force, delta_col='Δ', max_delta=1000, bins=80)

plot_error_rate_with_tau_markers(df_moves, delta_col='Δ', max_delta=1000, bins=50)

# 不同Remain Time下，分别拟合不同组选手的三参数公式曲线

In [ ]:
df_moves['Remain_Time'].describe()

In [ ]:
plt.figure(figsize = (8,5))
plt.hist(df_moves['Remain_Time'], bins=50)
plt.xlabel('Remain_Time')
plt.ylabel('Count')
plt.show()

# Remain Time随P(Optilmal)和 P(error) 的双坐标轴版本    每个图只专门选一组选手画
### 论文里放两三张图和表格应该就行了吧 （x0列需要加粗）

问题： error区没有图例；还有error区的纵坐标希望能加一个自动调节lim的功能，对不同组找到适合自己的

In [ ]:
def model_3param(x, A, tau, x0):
    x_safe = np.clip(-(x - x0) / tau, -500, 500)
    return A / (1 + np.exp(x_safe))

def get_empirical_with_ci(df, delta_min=15, delta_max=800, bin_size=10, min_count=20):
    if df.empty: return None
    # 彻底过滤 15cp 以下的数据
    df_filtered = df[(df['Δ'] >= delta_min) & (df['Δ'] <= delta_max)].copy()
    if df_filtered.empty: return None
    
    df_filtered['Δ_bin'] = (df_filtered['Δ'] // bin_size) * bin_size
    
    agg = df_filtered.groupby('Δ_bin').agg(
        P_opt=('Is_Optimal', 'mean'),
        P_err=('Is_Error', 'mean'),
        Count=('Is_Optimal', 'count')
    ).reset_index()
    
    agg = agg[agg['Count'] >= min_count]
    # 95% 置信区间
    agg['opt_ci'] = 1.96 * np.sqrt(agg['P_opt']*(1-agg['P_opt'])/agg['Count'])
    agg['err_ci'] = 1.96 * np.sqrt(agg['P_err']*(1-agg['P_err'])/agg['Count'])
    
    return agg.rename(columns={'Δ_bin': 'Δ'}).sort_values('Δ')

remain_time_list = [[5, 20], [50, 80], [100, 140]]
group_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']
blue_palette = ['#084594', '#2171b5', '#6baed6'] 
orange_palette = ['#8c2d04', '#d94801', '#f16913']

all_summary_results = []

# 绘图与拟合
for g in group_order:
    df_g = df_moves[(df_moves['ELO_Group'] == g) & (df_moves['Is_Forced'] == 0)]
    if df_g.empty: continue
        
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    # 拟合参考线从横轴为 15 开始
    x_ref = np.linspace(15, 800, 400) 
    max_err_val = 0
    
    for idx, time_range in enumerate(remain_time_list):
        t_start, t_end = time_range
        label = f"{t_start}-{t_end}s"
        
        df_sub = df_g[(df_g['Remain_Time'] >= t_start) & (df_g['Remain_Time'] <= t_end)]
        # 获取 15cp 起的数据
        emp_df = get_empirical_with_ci(df_sub, delta_min=15, bin_size=10, min_count=20)
        
        if emp_df is None or emp_df.empty: continue
        
        x_d = emp_df['Δ'].values
        y_o, y_e = emp_df['P_opt'].values, emp_df['P_err'].values
        ci_o, ci_e = emp_df['opt_ci'].values, emp_df['err_ci'].values
        
        try:
            # 拟合
            popt, pcov = curve_fit(model_3param, x_d, y_o, p0=[0.85, 180, 50], bounds=([0.4, 50, -100], [1.0, 400, 400]))
            perr = np.sqrt(np.diag(pcov)) 
            
            # 绘图
            ax1.plot(x_ref, model_3param(x_ref, *popt), color=blue_palette[idx], lw=2.5, zorder=4, label=f'P(Opt) {label} ($x_0$={popt[2]:.1f})')
            ax1.fill_between(x_d, y_o - ci_o, y_o + ci_o, color=blue_palette[idx], alpha=0.15, edgecolor='none', zorder=1)
            ax1.scatter(x_d, y_o, color=blue_palette[idx], s=12, alpha=0.3, zorder=2)
            
            all_summary_results.append({
                'RemainTime': label, 'ELO': g, 'A': popt[0], 'A_err': perr[0], 'tau': popt[1], 'tau_err': perr[1], 'x0': popt[2], 'x0_err': perr[2],
                'RMSE': np.sqrt(mean_squared_error(y_o, model_3param(x_d, *popt)))
            })
        except: pass
        
        # P(Error)
        ax2.plot(x_d, y_e, color=orange_palette[idx], alpha=0.8, linestyle='--', lw=1.5, label=f'P(Err) {label}')
        ax2.fill_between(x_d, y_e - ci_e, y_e + ci_e, color=orange_palette[idx], alpha=0.08, edgecolor='none')
        current_max = np.nanmax(y_e + ci_e)
        if current_max > max_err_val:
            max_err_val = current_max

    ax1.set_xlim(15, 800)
    # 横坐标刻度从 15算起，后面是整百
    custom_ticks = [15, 100, 200, 300, 400, 500, 600, 700, 800]
    ax1.set_xticks(custom_ticks)
    
    ax1.set_title(f'Cognitive Performance: ELO {g}', fontsize=14, fontweight='bold')
    ax1.set_xlabel(r'Decision Margin $\Delta$ (cp)', fontsize=12)
    ax1.set_ylabel('P(Optimal Move)', color=blue_palette[0], fontsize=12)
    ax2.set_ylabel('P(Error)', color=orange_palette[0], fontsize=12)
    
    ax1.set_ylim(0.25, 1.05)
    if max_err_val > 0:
        ax2.set_ylim(0, max_err_val*1.1)
    else:
        ax2.set_ylim(0, 0.40)
    ax1.grid(True, linestyle='--', alpha=0.3)
    ax1.legend(loc='upper left', fontsize=8, ncol=1)
    ax2.legend(loc='upper left', bbox_to_anchor=(0.25, 1), fontsize=8, ncol=1)

    plt.tight_layout()
    plt.show()

# 结果表格
res_df = pd.DataFrame(all_summary_results)
print("\n" + "="*80)
print(f"{'RemainTime':<12} {'ELO':<12} {'A(±err)':<15} {'tau(±err)':<15} {'x0(±err)':<15} {'RMSE':<8}")
print("-" * 80)
for _, r in res_df.iterrows():
    a_str = f"{r['A']:.3f}({r['A_err']:.3f})"
    t_str = f"{r['tau']:.1f}({r['tau_err']:.1f})"
    x_str = f"{r['x0']:.1f}({r['x0_err']:.1f})"
    print(f"{r['RemainTime']:<12} {r['ELO']:<12} {a_str:<15} {t_str:<15} {x_str:<15} {r['RMSE']:.4f}")

In [ ]:
def model_3param(x, A, tau, x0):
    x_safe = np.clip(-(x - x0) / tau, -500, 500)
    return A / (1 + np.exp(x_safe))

# --- 2. 增强版经验指标提取 (带置信区间计算) ---
def get_empirical_with_ci(df, delta_min=10, delta_max=800, bin_size=20, min_count=20):
    if df.empty: return None
    df = df[(df['Δ'] > delta_min) & (df['Δ'] <= delta_max)].copy()
    df['Δ_bin'] = (df['Δ'] // bin_size) * bin_size
    
    # 聚合：均值、计数
    agg = df.groupby('Δ_bin').agg(
        P_opt=('Is_Optimal', 'mean'),
        P_err=('Is_Error', 'mean'),
        Count=('Is_Optimal', 'count')
    ).reset_index()
    
    # 过滤噪音
    agg = agg[agg['Count'] >= min_count]
    
    # 计算 95% CI 误差: 1.96 * sqrt(p(1-p)/n)
    agg['opt_ci'] = 1.96 * np.sqrt(agg['P_opt']*(1-agg['P_opt'])/agg['Count'])
    agg['err_ci'] = 1.96 * np.sqrt(agg['P_err']*(1-agg['P_err'])/agg['Count'])
    
    return agg.rename(columns={'Δ_bin': 'Δ'}).sort_values('Δ')

# --- 3. 颜色与配置 ---
remain_time_list = [[5, 20], [50, 80], [100, 140]]
group_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']

# 调色盘：学术风
blue_palette = ['#084594', '#2171b5', '#6baed6']  # P(Optimal)
orange_palette = ['#8c2d04', '#d94801', '#f16913'] # P(Error) 橙色调

all_summary_results = []

# --- 4. 绘图与拟合 ---
for g in group_order:
    df_g = df_moves[(df_moves['ELO_Group'] == g) & (df_moves['Is_Forced'] == 0)]
    if df_g.empty: continue
        
    fig, ax1 = plt.subplots(figsize=(11, 6.5))
    ax2 = ax1.twinx()
    
    x_ref = np.linspace(0, 800, 500)
    
    for idx, time_range in enumerate(remain_time_list):
        t_start, t_end = time_range
        label = f"{t_start}-{t_end}s"
        
        # 数据切片
        df_sub = df_g[(df_g['Remain_Time'] >= t_start) & (df_g['Remain_Time'] <= t_end)]
        
        # 提取经验点 (bin_size调小可以让点更多，10是一个较好的平衡点)
        emp_df = get_empirical_with_ci(df_sub, bin_size=10, min_count=20)
        if emp_df is None or emp_df.empty: continue
        
        x_d = emp_df['Δ'].values
        y_o, y_e = emp_df['P_opt'].values, emp_df['P_err'].values
        ci_o, ci_e = emp_df['opt_ci'].values, emp_df['err_ci'].values
        
        # --- 3-P 拟合 P(Optimal) ---
        try:
            popt, pcov = curve_fit(model_3param, x_d, y_o, 
                                   p0=[0.85, 180, 50], 
                                   bounds=([0.4, 50, -100], [1.0, 400, 400]))
            perr = np.sqrt(np.diag(pcov)) # 提取 A_err, tau_err, x0_err
            
            # 绘制左轴：拟合曲线 + 带有误差棒的散点
            ax1.plot(x_ref, model_3param(x_ref, *popt), color=blue_palette[idx], 
                     lw=2, zorder=3, label=f'P(Opt) {label} ($x_0$={popt[2]:.1f})')
            # 仅在关键步长（如每3个点）画误差棒，避免画面太乱
            # ax1.errorbar(x_d[::2], y_o[::2], yerr=ci_o[::2], fmt='o', color=blue_palette[idx], ecolor=blue_palette[idx], elinewidth=1, capsize=2, markersize=4, alpha=0.5, zorder=2)
            
            all_summary_results.append({
                'RemainTime': label, 'ELO': g, 
                'A': popt[0], 'A_err': perr[0],
                'tau': popt[1], 'tau_err': perr[1],
                'x0': popt[2], 'x0_err': perr[2],
                'RMSE': np.sqrt(mean_squared_error(y_o, model_3param(x_d, *popt)))
            })
        except: pass
        
        # --- 绘制右轴：P(Error) 经验曲线 ---
        ax2.plot(x_d, y_e, color=orange_palette[idx], alpha=0.6, linestyle=(0, (3, 1, 1, 1)), 
                 label=f'P(Err) {label}')
        # 绘制 Error 的散点
        # ax2.scatter(x_d[::3], y_e[::3], color=orange_palette[idx], s=15, marker='x', alpha=0.4)
        ax1.fill_between(x_d, y_o - ci_o, y_o + ci_o, color=blue_palette[idx], alpha=0.15, edgecolor='none')
        ax1.plot(x_ref, model_3param(x_ref, *popt), color=blue_palette[idx], lw=2.5, label=f'P(Opt) {label}')
        ax2.fill_between(x_d, y_e - ci_e, y_e + ci_e, color=orange_palette[idx], alpha=0.1, edgecolor='none')
        # 使用样条插值或者平滑线画 P-Error 趋势
        ax2.plot(x_d, y_e, color=orange_palette[idx], alpha=0.8, linestyle='--', lw=1.5)

    # 样式精修
    ax1.set_title(f'Cognitive Performance Under Time Pressure | ELO: {g}', fontsize=14, pad=15)
    ax1.set_xlabel(r'Decision Margin $\Delta$ (cp)', fontsize=12)
    ax1.set_ylabel('P(Optimal Move)', color=blue_palette[0], fontsize=12)
    ax2.set_ylabel('P(Error)', color=orange_palette[0], fontsize=12)
    
    ax1.set_xlim(0, 800)
    ax1.set_ylim(0.2, 1.05)
    ax2.set_ylim(0, 0.40)
    
    ax1.grid(True, linestyle='--', alpha=0.3)
    
    # 图例合并
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8, ncol=2, frameon=True)

    plt.tight_layout()
    plt.show()

# --- 5. 汇总完整参数表 ---
res_df = pd.DataFrame(all_summary_results)
print("\n" + "="*80)
print(f"{'RemainTime':<12} {'ELO':<12} {'A(±err)':<15} {'tau(±err)':<15} {'x0(±err)':<15} {'RMSE':<8}")
print("-" * 80)
for _, r in res_df.iterrows():
    a_str = f"{r['A']:.3f}({r['A_err']:.3f})"
    t_str = f"{r['tau']:.1f}({r['tau_err']:.1f})"
    x_str = f"{r['x0']:.1f}({r['x0_err']:.1f})"
    print(f"{r['RemainTime']:<12} {r['ELO']:<12} {a_str:<15} {t_str:<15} {x_str:<15} {r['RMSE']:.4f}")

难度门槛 ($x_0$) 的主导作用：    
研究最显著的发现是：在所有技术水平组中，难度门槛 $x_0$ 均随思考时间的缩短而表现出严格的单调递增。例如，在顶级组 ($\text{ELO} \ge 3000$) 中，$x_0$ 从 $13.5$ cp（慢棋）增加到了 $25.1$ cp（快棋）。这种趋势在低分段组中尤为剧烈。这一证据表明，时间压力的核心影响在于大幅提高了“认知准入门槛”，即棋手需要更大的局面优势才能激活逻辑思维并做出最优决策。    
认知精度与噪音 ($\tau$)：    
缩放参数 $\tau$ 随时间缩短而普遍增大。$\tau$ 的上升反映了认知精度的下降和决策噪音的增加。在高时间压力下，$P(\text{Optimal})$ 曲线的坡度变得更加平缓，这意味着即使局面难度超过了门槛 $x_0$，棋手选择引擎推荐走法的可靠性也会因时间匮乏而受损。     
准确率天花板 ($A$) 的渐进稳定性：     
与 $x_0$ 和 $\tau$ 不同，准确率上限 $A$ 表现出显著的渐进稳定性，基本维持在 $0.85$ 至 $0.91$ 之间。尽管在极短时间区间（$5\text{-}20$s）内观察到 $A$ 值的微小波动（有时略高），但这些波动通常伴随着更大的标准差。这表明 $A$ 更多地代表了人类直觉的极限，而非系统性的表现提升。$A$ 的稳健性意味着棋手的巅峰执行能力是一种内在特质，在很大程度上不受外部时间压力的干扰。

### 较为传统的，每个图都对应一个remain time区间然后把所有分组放进去，但我觉得不太好

In [ ]:
def model_3param(x, A, tau, x0):
    """3参数逻辑回归模型"""
    x_safe = np.clip(-(x - x0) / tau, -500, 500)
    return A / (1 + np.exp(x_safe))

def empirical_curve_refined(df, delta_min=10, delta_max=800, min_count=50, bin_size=20):
    """从过滤后的数据中提取经验准确率曲线"""
    if df.empty: return None
    
    # 分箱处理
    df = df[(df['Δ'] > delta_min) & (df['Δ'] <= delta_max)].copy()
    df['Δ_bin'] = (df['Δ'] // bin_size) * bin_size
    
    # 过滤低频分箱
    counts = df.groupby('Δ_bin')['Is_Optimal'].count()
    valid_bins = counts[counts >= min_count].index
    df = df[df['Δ_bin'].isin(valid_bins)]
    
    if df.empty: return None
    
    # 聚合
    prob_df = df.groupby('Δ_bin')['Is_Optimal'].agg(
        P_Optimal='mean',
        Count='count'
    ).reset_index().rename(columns={'Δ_bin': 'Δ'})
    
    return prob_df.sort_values('Δ')


remain_time_list = [[5, 20], [50, 65], [90, 105]]
group_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']
distinct_palette = sns.color_palette("viridis", len(group_order))

all_summary_records = []


for time_range in remain_time_list:
    t_start, t_end = time_range
    time_label = f"RT_{t_start}-{t_end}"
    print(f"\n>>> 正在处理时间区间: {time_label}")
    
    # 预过滤当前时间区间的数据
    df_time = df_moves[
        (df_moves['Remain_Time'] >= t_start) & 
        (df_moves['Remain_Time'] <= t_end) & 
        (df_moves['Is_Forced'] == 0)
    ]
    
    plt.figure(figsize=(12, 7))
    plt.title(f'3-P Model Fitting (Unweighted) | RemainTime: {t_start}-{t_end}s', fontsize=14, fontweight='bold')
    
    x_ref = np.linspace(0, 800, 400)
    
    for i, g in enumerate(group_order):
        # 提取当前等级的经验数据
        df_group = df_time[df_time['ELO_Group'] == g]
        prob_df = empirical_curve_refined(df_group, bin_size=5, min_count=30) # 时间分段后数据变稀疏，调低min_count
        
        if prob_df is None or prob_df.empty:
            continue
            
        x_data = prob_df['Δ'].values
        y_data = prob_df['P_Optimal'].values
        
        # --- 拟合 ---
        try:
            # P0 猜想值: A=0.85, tau=180, x0=50
            popt, pcov = curve_fit(model_3param, x_data, y_data, 
                                   p0=[0.85, 180, 50], 
                                   bounds=([0.5, 1, -200], [1.0, 1000, 400]))
            
            # 计算误差
            perr = np.sqrt(np.diag(pcov))
            y_pred = model_3param(x_data, *popt)
            rmse = np.sqrt(mean_squared_error(y_data, y_pred))
            
            # 保存结果
            all_summary_records.append({
                'RemainTime_Range': time_label,
                'ELO_Group': g,
                'A': popt[0], 'A_err': perr[0],
                'tau': popt[1], 'tau_err': perr[1],
                'x0': popt[2], 'x0_err': perr[2],
                'RMSE': rmse,
                'Sample_Size': len(df_group)
            })
            
            # 绘图
            color = distinct_palette[i]
            plt.scatter(x_data, y_data, color=color, alpha=0.3, s=15, edgecolors='none')
            plt.plot(x_ref, model_3param(x_ref, *popt), color=color, lw=2, label=f"{g} (x0={popt[2]:.1f})")
            
        except Exception as e:
            print(f"拟合失败 [{g}]: {e}")

    # 图形装饰
    plt.xlabel('Decision Margin Δ (cp)', fontsize=12)
    plt.ylabel('P(Optimal Move)', fontsize=12)
    plt.ylim(0.2, 1.05)
    plt.xlim(0, 800)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(loc='lower right', title="ELO Groups", fontsize=9)
    plt.tight_layout()
    plt.show()

# --- 4. 生成汇总表 ---

res_df = pd.DataFrame(all_summary_records)
# 整理列顺序
res_df = res_df[['RemainTime_Range', 'ELO_Group', 'A', 'A_err', 'tau', 'tau_err', 'x0', 'x0_err', 'RMSE']]

print("\n" + "="*50)
print("FINAL MULTI-DIMENSIONAL FITTING SUMMARY")
print("="*50)
print(res_df.to_string(index=False, formatters={
    'A': '{:.3f}'.format, 'A_err': '{:.3f}'.format,
    'tau': '{:.1f}'.format, 'tau_err': '{:.1f}'.format,
    'x0': '{:.1f}'.format, 'x0_err': '{:.1f}'.format,
    'RMSE': '{:.5f}'.format
}))

# 画思考时间MoveTime对走出最优解概率和错误率的影响关系
### 每步思考时间越久到底是更容易走出最优解还是更容易错 是否有关联

In [ ]:
print(df_moves['Move_Time'].describe())
bins = np.linspace(0, 50, 101)  
plt.figure(figsize = (8,5))
plt.hist(df_moves['Move_Time'], bins=bins)
plt.xlim(0,50)
plt.xlabel('Move_Time')
plt.ylabel('Count')
plt.show()

In [ ]:
# 定义边界和标签
bins = [0, 1.5, 5.0, np.inf]
labels = ['Instant', 'Rapid', 'Slow']
# 思考时间分箱
df_moves['Move_Time_Group'] = pd.cut(df_moves['Move_Time'], bins=bins, labels=labels)
# 检查各组样本量，确保 6200 万行分得均匀
print(df_moves['Move_Time_Group'].value_counts())

问题： error区没有图例；还有error区的纵坐标希望能加一个自动调节lim的功能，对不同组找到适合自己的

In [ ]:
labels = ['Instant', 'Rapid', 'Slow']

colors_opt = ['#6baed6', '#2171b5', '#084594'] 
colors_err = ['#fdbe85', '#e6550d', '#a63603']
min_count_dict = {
    'Instant': 120, 
    'Rapid': 80, 
    'Slow': 10  # 降低 Slow 的准入门槛
}

all_summary = []
group_order = ['>=3000', '2800-3000', '2600-2800', '2400-2600', '2200-2400', '2000-2200', '<2000']

# 绘图
for g in group_order:
    df_g = df_moves[(df_moves['ELO_Group'] == g) & (df_moves['Is_Forced'] == 0)]
    if df_g.empty: continue

    fig, ax1 = plt.subplots(figsize=(11, 6.5))
    ax2 = ax1.twinx()
    x_ref = np.linspace(15, 800, 500)
    max_err_val = 0 
    
    print(f"正在处理 ELO: {g} ...")

    for idx, mt_label in enumerate(labels):
        df_sub = df_g[df_g['Move_Time_Group'] == mt_label]
        
        # 动态调整 bin_size 
        # 如果是 Slow 组，把箱子从 10 扩大到 30，更容易凑够样本数
        current_bin = 30 if mt_label == 'Slow' else 10
        curr_min = min_count_dict.get(mt_label, 50)
        
        # 经验数据提取 (15cp起)
        emp_df = get_empirical_with_ci(df_sub, delta_min=15, delta_max=800, bin_size=current_bin, min_count=curr_min)
        
        if emp_df is None or emp_df.empty:
            if mt_label == 'Slow':
                print(f"  [警告] {g} 的 Slow 组数据依然不足，建议检查该组原始样本量。")
            continue
        
        x_d, y_o, y_e = emp_df['Δ'].values, emp_df['P_opt'].values, emp_df['P_err'].values
        ci_o, ci_e = emp_df['opt_ci'].values, emp_df['err_ci'].values
        
        # 追踪右轴最大值
        max_err_val = max(max_err_val, np.max(y_e + ci_e))
        
        # 拟合与绘图
        try:
            # 增加 maxfev 确保稀疏数据也能收敛
            popt, pcov = curve_fit(model_3param, x_d, y_o, p0=[0.88, 150, 40],
                                   bounds=([0.4, 30, -100], [1.0, 500, 400]), maxfev=5000)
            perr = np.sqrt(np.diag(pcov))
            
            # A. 左轴：P(Optimal) 实线 + 阴影
            ax1.plot(x_ref, model_3param(x_ref, *popt), color=colors_opt[idx], 
                     lw=2.5, alpha=1.0, zorder=10,
                     label=f'P(Opt) {mt_label} ($x_0$={popt[2]:.1f})')
            ax1.fill_between(x_d, y_o - ci_o, y_o + ci_o, color=colors_opt[idx], alpha=0.15)
            
            # B. 右轴：P(Error) 虚线 + 阴影
            ax2.plot(x_d, y_e, '--', color=colors_err[idx], alpha=0.9, lw=1.8, label=f'P(Err) {mt_label}')
            ax2.fill_between(x_d, y_e - ci_e, y_e + ci_e, color=colors_err[idx], alpha=0.08)
            
            all_summary.append({
                'ELO': g, 'MT': mt_label, 'A': popt[0], 'A_err': perr[0], 
                'tau': popt[1], 'tau_err': perr[1], 'x0': popt[2], 'x0_err': perr[2],
                'RMSE': np.sqrt(mean_squared_error(y_o, model_3param(x_d, *popt)))
            })
        except Exception as e:
            print(f"  拟合失败: {g} - {mt_label} | 原因: {e}")
            continue

    ax1.set_xlim(15, 800)
    ax1.set_xticks([15, 100, 200, 400, 600, 800])
    ax1.set_ylim(0.25, 1.05)
    
    # 动态调整右轴 Y 量程
    ax2.set_ylim(0, np.ceil(max_err_val * 20) / 20 + 0.05) 

    ax1.set_title(f'Move Thinking Time Impact | ELO: {g}', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel(r'Decision Margin $\Delta$ (cp)', fontsize=12)
    ax1.set_ylabel('P(Optimal Move)', color=colors_opt[-1], fontsize=12, fontweight='bold')
    ax2.set_ylabel('P(Error)', color=colors_err[-1], fontsize=12, fontweight='bold')

    ax1.grid(True, linestyle='--', alpha=0.3)
    
    h1, l1 = ax1.get_legend_handles_labels()
    ax1.legend(h1, l1, loc='upper left', fontsize=8, ncol=1, framealpha=0.9, edgecolor='gray')
    h2, l2 = ax2.get_legend_handles_labels()
    ax2.legend(h2, l2, loc='upper left', bbox_to_anchor = (0.20, 1), fontsize=8, ncol=1, framealpha=0.9, edgecolor='gray')

    plt.tight_layout()
    plt.show()

# 结果汇总表
final_df = pd.DataFrame(all_summary)
print(final_df.to_string(index=False))

In [ ]:
pd.set_option('display.max_columns', None)
df_moves.head()

## 游戏不同阶段的剩余时间和最终输赢的关系

In [ ]:
def plot_win_evolution(df, progress_list, time_bins=None, time_labels=None, add_ci=True, add_sample_count=True):
    if time_bins is None:
        time_bins = [0, 10, 30, 60, 120, np.inf]
    if time_labels is None:
        time_labels = ['0-10s', '10-30s', '30-60s', '60-120s', '>120s']

    res_map = {'win': 1.0, 'draw': 0.5, 'lose': 0.0}
    elo_order = ['2000-2200', '2200-2400', '2400-2600', '2600-2800', '2800-3000', '>=3000']
    distinct_palette = sns.color_palette("viridis", len(elo_order))
    color_map = dict(zip(elo_order, distinct_palette))
    all_results = []

    for p in progress_list:
        print(f"\n======================================")
        print(f" 进度 {p*100:.0f}% 开局/中局/残局分析")        
        idx = (df['Progress'] - p).abs().groupby(df['uid']).idxmin()
        df_snap = df.loc[idx].copy()
        df_snap['Win_Score'] = df_snap['Result'].map(res_map).astype(float)
        df_snap['Time_Bucket'] = pd.cut(df_snap['Remain_Time'], bins=time_bins, labels=time_labels, include_lowest=True)
        def calc_stats(group):
            mean = group['Win_Score'].mean()
            n = len(group)
            se = stats.sem(group['Win_Score']) if n >= 2 else 0
            ci = stats.t.interval(0.95, n-1, loc=mean, scale=se) if n >=2 else (mean, mean)
            return pd.Series([mean, n, ci[0], ci[1]], index=['Win_Score','Count','CI_Low','CI_High'])

        # 计算统计量
        def calc_stats(group):
            mean = group['Win_Score'].mean()
            n = len(group)
            se = stats.sem(group['Win_Score']) if n >= 2 else 0
            ci = stats.t.interval(0.95, n-1, loc=mean, scale=se) if n >=2 else (mean, mean)
            return pd.Series([mean, n, ci[0], ci[1]], index=['Win_Score','Count','CI_Low','CI_High'])
        
        plot_data = df_snap.groupby(['Time_Bucket', 'ELO_Group'], observed=True).apply(calc_stats).reset_index()

        # 计算胜率提升
        gain_results = []
        for elo in elo_order:
            group = plot_data[plot_data['ELO_Group'] == elo]
            if group.empty: continue

            # 固定取：0-10s 和 >120s
            p0 = group[group['Time_Bucket'] == '0-10s']['Win_Score'].values
            p1 = group[group['Time_Bucket'] == '>120s']['Win_Score'].values

            if len(p0) > 0 and len(p1) > 0:
                win_gain = float(p1[0] - p0[0])
                gain_results.append({
                    'ELO_Group': elo,
                    'Progress': p,
                    'WinProb_0-10s': round(float(p0[0]),3),
                    'WinProb_>120s': round(float(p1[0]),3),
                    'WinProb_GAIN': round(win_gain,3)
                })

        # 输出当前进度的计算结果
        gain_df = pd.DataFrame(gain_results).sort_values('WinProb_GAIN', ascending=False)
        all_results.extend(gain_results)
        
        print(f"\n 时间优势转化胜率能力排名:")
        print(gain_df[['ELO_Group','WinProb_0-10s','WinProb_>120s','WinProb_GAIN']].to_string(index=False))

        best = gain_df.iloc[0]
        print(f"\n 转化能力最强：{best['ELO_Group']}， win prob increases {best['WinProb_GAIN']:.3f}")

        # 绘图部分
        plt.figure(figsize=(12,7))
        for elo in elo_order:
            gd = plot_data[plot_data['ELO_Group'] == elo]
            if gd.empty: continue
            sns.lineplot(data=gd, x='Time_Bucket', y='Win_Score', color=color_map[elo], marker='o', linewidth=2.5, label=elo)
            if add_ci:
                plt.fill_between(gd['Time_Bucket'], gd['CI_Low'], gd['CI_High'], color=color_map[elo], alpha=0.15)
            if add_sample_count:
                for _, row in gd.iterrows():
                    plt.text(
                        row['Time_Bucket'], 
                        row['Win_Score'] + 0.02, 
                        f"n={int(row['Count'])}", 
                        ha='center', va='bottom', fontsize=7, color=color_map[elo]
                    )
        plt.axhline(0.5, ls='--', color='gray', alpha=0.6)
        plt.title(f'Win Probability vs Time at {p*100:.0f}% Progress', fontweight='bold', fontsize=15)
        plt.xlabel('Remaining Time')
        plt.ylabel('Average Win Probability')
        plt.ylim(0, 1)
        plt.grid(alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
        plt.tight_layout()
        plt.show()

    # 完整表格
    final_df = pd.DataFrame(all_results).round(3)
    print("\n\n" + "="*60)
    print("Final Results")
    print("="*60)
    print(final_df.to_string(index=False))

    return final_df

my_milestones = [0.1, 0.3, 0.5, 0.7, 0.8, 0.95]
df_result = plot_win_evolution(df_moves, my_milestones)

In [ ]:
def plot_win_evolution(df, progress_list, time_bins=None, time_labels=None, add_ci=True, add_sample_count=True):
    if time_bins is None:
        time_bins = [0, 10, 30, 60, 120, np.inf]
    if time_labels is None:
        time_labels = ['0-10s', '10-30s', '30-60s', '60-120s', '>120s']

    res_map = {'win': 1.0, 'draw': 0.5, 'lose': 0.0}
    elo_order = ['2000-2200', '2200-2400', '2400-2600', '2600-2800', '2800-3000', '>=3000']
    
    distinct_palette = sns.color_palette("viridis", len(elo_order))
    color_map = dict(zip(elo_order, distinct_palette))
    all_results = []

    def calc_stats(group):
        mean = group['Win_Score'].mean()
        n = len(group)
        # 计算置信区间
        se = stats.sem(group['Win_Score']) if n >= 2 else 0
        ci = stats.t.interval(0.95, n-1, loc=mean, scale=se) if n >= 2 else (mean, mean)
        return pd.Series([mean, n, ci[0], ci[1]], index=['Win_Score', 'Count', 'CI_Low', 'CI_High'])

    for p in progress_list:
        print(f"\n======================================")
        print(f" 进度 {p*100:.0f}% 深度分析 (双视角采样)")
        
        # 加入 observed=True 解决报错
        # 我们先算距离，再分组找最小索引
        dist_series = (df['Progress'] - p).abs()
        idx = dist_series.groupby([df['uid'], df['Color']], observed=True).idxmin()
        
        df_snap = df.loc[idx].copy()
        
        # 容差过滤：只看目标进度附近的点
        df_snap = df_snap[(df_snap['Progress'] - p).abs() < 0.1]
        
        if df_snap.empty:
            print(f" 进度 {p*100:.0f}% 无符合条件样本，跳过。")
            continue

        df_snap['Win_Score'] = df_snap['Result'].map(res_map).astype(float)
        df_snap['Time_Bucket'] = pd.cut(df_snap['Remain_Time'], bins=time_bins, labels=time_labels, include_lowest=True)
        
        # 计算统计量
        plot_data = df_snap.groupby(['Time_Bucket', 'ELO_Group'], observed=True).apply(calc_stats).reset_index()

        # 计算并打印排名 (Gain Calculation)
        gain_results = []
        for elo in elo_order:
            group = plot_data[plot_data['ELO_Group'] == elo]
            if group.empty: continue
            p0 = group[group['Time_Bucket'] == '0-10s']['Win_Score'].values
            p1 = group[group['Time_Bucket'] == '>120s']['Win_Score'].values
            if len(p0) > 0 and len(p1) > 0:
                win_gain = float(p1[0] - p0[0])
                gain_results.append({'ELO_Group': elo, 'Progress': p, 
                                     'WinProb_0-10s': round(float(p0[0]), 3), 
                                     'WinProb_>120s': round(float(p1[0]), 3), 
                                     'WinProb_GAIN': round(win_gain, 3)})

        gain_df = pd.DataFrame(gain_results).sort_values('WinProb_GAIN', ascending=False)
        all_results.extend(gain_results)
        
        if not gain_df.empty:
            print(f"\n 时间优势转化胜率能力排名:")
            print(gain_df[['ELO_Group','WinProb_0-10s','WinProb_>120s','WinProb_GAIN']].to_string(index=False))

        # 绘图
        plt.figure(figsize=(10, 6))
        for elo in elo_order:
            gd = plot_data[plot_data['ELO_Group'] == elo]
            if gd.empty: continue
            
            sns.lineplot(data=gd, x='Time_Bucket', y='Win_Score', color=color_map[elo], marker='o', linewidth=2.5, label=elo)
            if add_ci:
                plt.fill_between(gd['Time_Bucket'], gd['CI_Low'], gd['CI_High'], color=color_map[elo], alpha=0.1)
            if add_sample_count:
                for _, row in gd.iterrows():
                    plt.text(row['Time_Bucket'], row['Win_Score'] + 0.01, f"n={int(row['Count'])}", 
                             ha='center', va='bottom', fontsize=7, color=color_map[elo])

        plt.axhline(0.5, ls='--', color='gray', alpha=0.5)
        plt.title(f'Win Probability vs Time at {p*100:.0f}% (Dual-Perspective)', fontweight='bold')
        plt.ylim(0, 1)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

    return pd.DataFrame(all_results).round(3)

my_milestones = [0.1, 0.3, 0.5, 0.7, 0.8, 0.95]
final_results_df = plot_win_evolution(df_moves, my_milestones)

### 在中残局阶段，时间优势越大最后更有可能获胜
### elo高的选手组，其胜率随剩余时间增加的上升幅度更大。高水平棋手能够更有效地将时间优势转化为最终获胜概率，而低等级棋手对时间优势的利用效率相对较低

In [ ]:
def plot_win_probability_dual_perspective(df, progress_bins=12, min_sample=500):
    res_map = {'win': 1.0, 'draw': 0.5, 'lose': 0.0}
    df_temp = df.copy()
    
    # 进度分箱
    df_temp['Progress_Bin'] = pd.cut(df_temp['Progress'], bins=progress_bins)
    
    # 双视角去重
    # 按 uid (对局) 和 Color (选手) 分组，每人在每个进度桶只取一个快照
    # 保证了样本中：赢家总数 == 输家总数
    df_plot = df_temp.groupby(['uid', 'Color', 'Progress_Bin'], observed=True).first().reset_index()
    
    # 计算胜率数值和坐标
    df_plot['Win_Score'] = df_plot['Result'].map(res_map).astype(float)
    df_plot['Progress_Mid'] = df_plot['Progress_Bin'].apply(lambda x: x.mid).astype(float)

    # 时间分桶
    time_bins = [0, 10, 30, 60, 120, np.inf]
    time_labels = ['0-10s', '10-30s', '30-60s', '60-120s', '>120s']
    df_plot['Time_Bucket'] = pd.cut(df_plot['Remain_Time'], bins=time_bins, labels=time_labels, include_lowest=True)

    # 统计计算
    def calc_stats(group):
        mean = group['Win_Score'].mean()
        n = len(group)
        se = stats.sem(group['Win_Score']) if n >= 2 else 0
        ci_low, ci_high = stats.t.interval(0.95, n-1, loc=mean, scale=se) if n >= 2 else (mean, mean)
        return pd.Series([mean, ci_low, ci_high, n], index=['Win_Score', 'CI_Low', 'CI_High', 'Count'])

    # 分组绘图
    elo_order = ['2000-2200', '2200-2400', '2400-2600', '2600-2800', '2800-3000', '>=3000']
    available_elos = [e for e in elo_order if e in df_plot['ELO_Group'].unique()]

    n_plots = len(available_elos)
    cols = 2
    rows = (n_plots + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
    axes = axes.flatten()
    colors = sns.color_palette('viridis', len(time_labels))

    for idx, elo in enumerate(available_elos):
        ax = axes[idx]
        sub_df = df_plot[df_plot['ELO_Group'] == elo]
        
        # 可选择是否再次打印检查，lose 的比例是否正常，应该会回到 40%-50% 左右
        # print(f"ELO {elo} 胜率分布:\n", sub_df['Result'].value_counts(normalize=True))

        stats_df = sub_df.groupby(['Progress_Mid', 'Time_Bucket'], observed=True).apply(calc_stats).reset_index()
        stats_df = stats_df[stats_df['Count'] >= min_sample]

        for i, t in enumerate(time_labels):
            line_data = stats_df[stats_df['Time_Bucket'] == t].sort_values('Progress_Mid')
            if line_data.empty: continue

            ax.plot(line_data['Progress_Mid'], line_data['Win_Score'],
                    marker='o', markersize=4, linewidth=2.5, color=colors[i], label=t)
            ax.fill_between(line_data['Progress_Mid'], line_data['CI_Low'], line_data['CI_High'],
                            color=colors[i], alpha=0.15)

        ax.set_title(f'ELO: {elo}', fontweight='bold', fontsize=13)
        # ax.set_ylim(0, 1)
        ax.axhline(0.5, ls='--', color='gray', alpha=0.6)
        ax.legend(title='Remaining Time', fontsize=9)
        # ax.set_ylim(auto=True)

    plt.tight_layout()
    plt.show()

plot_win_probability_dual_perspective(df_moves, progress_bins=10, min_sample=500)